# Total-weight Dowker duality

This notebook illustrates **Theorem 3.3** from
[arXiv:2405.15592](https://arxiv.org/abs/2405.15592):

> For a relation $R \subseteq X \times Y$, there is a weak equivalence of filtrations
> $$|\mathcal{D}(X, Y, R)_\bullet| \simeq |\mathcal{S}(\mathcal{D}(Y, X, R^T))_\bullet|$$
> where $\mathcal{S}$ denotes the subdivision filtration.

In particular, the **degree-0 persistence diagrams** of
$\mathcal{D}(X, Y, R)$ with the total-weight filtration and of
$\mathcal{S}(\mathcal{D}(Y, X, R^T))$ are identical.

In [ ]:
import numpy as np
import gudhi as gd
import matplotlib.pyplot as plt
from pyDowker import DowkerComplex

## Example relation

We use a $4 \times 5$ boolean relation matrix $R$ (rows = data points $X$,
columns = witnesses $Y$).  An entry of 1 means the pair is in the relation.

In [ ]:
relation = np.array(
    [[1, 1, 0, 0, 1],
     [1, 0, 0, 1, 1],
     [1, 0, 1, 1, 0],
     [0, 1, 1, 0, 0]], dtype=bool
)
print('R shape:', relation.shape, '  (rows=X, cols=Y)')

## Direct complex vs. subdivision complex

We construct three objects:

1. `cplx` — $\mathcal{D}(X, Y, R)$ with the total-weight filtration
2. `dual` — $\mathcal{D}(Y, X, R^T)$ with the total-weight filtration (for reference)
3. `subdiv` — $\mathcal{S}(\mathcal{D}(Y, X, R^T))$, the subdivision filtration
   (via `create_subdivision_simplex_tree`)

Theorem 3.3 predicts that `cplx` and `subdiv` have the same persistent homology.

In [ ]:
dc = DowkerComplex(relation)

cplx   = dc.create_simplex_tree(filtration='TotalWeight', max_dimension=5)
dual   = DowkerComplex(relation.T).create_simplex_tree(filtration='TotalWeight', max_dimension=5)
subdiv = dc.create_subdivision_simplex_tree(max_dimension=5)

cplx.compute_persistence()
dual.compute_persistence()
subdiv.compute_persistence()

print('D(X,Y,R) H0:       ', cplx.persistence_intervals_in_dimension(0))
print('D(Y,X,R^T) H0:     ', dual.persistence_intervals_in_dimension(0))
print('S(D(Y,X,R^T)) H0:  ', subdiv.persistence_intervals_in_dimension(0))

## Persistence diagrams

The diagrams of $\mathcal{D}(X, Y, R)$ (left) and $\mathcal{S}(\mathcal{D}(Y, X, R^T))$
(right) should be identical, confirming the duality theorem.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

gd.plot_persistence_diagram(cplx.persistence(), axes=axes[0])
axes[0].set_title(r'$\mathcal{D}(X,Y,R)$ — total-weight filtration')

gd.plot_persistence_diagram(subdiv.persistence(), axes=axes[1])
axes[1].set_title(r'$\mathcal{S}(\mathcal{D}(Y,X,R^T))$ — subdivision filtration')

plt.tight_layout()
plt.show()